# 08 · Bounded review loops

LoopAgent repeats its children until escalation or max_iterations. A deterministic reviewer can stop a loop without spending another model call.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** 1–2 Lite calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk.agents import BaseAgent, LoopAgent
from google.adk.events import Event, EventActions

class CheckDraft(BaseAgent):
    async def _run_async_impl(self, ctx):
        draft = ctx.session.state.get("draft", "")
        ok = "O1001" in draft and "INR" in draft
        yield Event(author=self.name, actions=EventActions(
            state_delta={"feedback": "Approved" if ok else "Include O1001 and INR explicitly."},
            escalate=ok))

writer = llm("draft_writer", "Write a two-sentence invoice note for O1001, total INR 4126.46. "
             "Apply feedback: {feedback}.", output_key="draft")
loop = LoopAgent(name="review_loop", sub_agents=[writer, CheckDraft(name="check_draft")], max_iterations=2)
lab = await make_lab(loop, state={"feedback": "Include the order ID and currency."})
await ask(lab, "Draft the note.")
print(lab.session.state)


For a model-driven reviewer, expose `google.adk.tools.exit_loop` as a tool and instruct it to call that tool only when the draft passes. This example uses explicit Python checks to make the stop condition reproducible.


## Try it

Require the invoice ID too. Watch the loop stop at its hard limit if the writer never meets the condition.


## Reference

[Official ADK documentation](https://adk.dev/). Shared configuration: `config.json`. No environment activation or login cells are needed.
